# Organized synthetic-data preprocessing pipeline

This notebook keeps each preprocessing step next to its reverse step through shared metadata dictionaries.

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

In [2]:
UNKNOWN_TOKENS = {"Unknown", "unknown"}
MISSING_TOKEN = "__MISSING__"

# Drop and restore later
restore_constants = {
    "Pneumonia::301": pd.NA,
    "Pleural Fluid::305": pd.NA,
    "Respiratory failure::308": pd.NA,
    "Pneumothorax::307": pd.NA,
    "Other respiratory complication::303": pd.NA,
    "Heart failure::287": pd.NA,
    "Deep venous thrombosis::285": pd.NA,
    "Portal Vein Thrombosis::289": pd.NA,
    "Pulmonary embolus::291": pd.NA,
    "Cerebrovascular lesion::294": pd.NA,
    "Cardiac arrest::296": pd.NA,
    "Hypertension::316": pd.NA,
    "Other cardiovascular complication::292": pd.NA,
    "Post dural-puncture headache::327": pd.NA,
    "Epidural hematoma or abscess::329": pd.NA,
    "Other EDA or spinal related complication::330": pd.NA,
}

restore_copies = {
    "Septic Shock::318": "Infected graft or prosthesis::314",
    "Sepsis::319": "Infected graft or prosthesis::314",
    "Urinary tract injury::328": "Anastomotic leak::324",
    "Postoperative excessive haemorrhage::338": "Intraoperative excessive haemorrhage::339",
    "Hematoma::336": "Intraoperative excessive haemorrhage::339",
}

date_cols = [
    "Date of last chemotherapy treatment (YYYY-MM-DD)::29",
    "Nasogastric tube reinserted date (YYYY-MM-DD)::461",
    "Date of Admission (YYYY-MM-DD)::17",
    "Termination of epidural analgesia (YYYY-MM-DD)::147",
    "Date of primary operation (YYYY-MM-DD)::53",
    "Stop of operation date (YYYY-MM-DD)::72",
    "First passage of flatus (YYYY-MM-DD)::127",
    "Termination of intravenous fluid infusion (YYYY-MM-DD)::108",
    "Nursed back to preoperative ADL ability (YYYY-MM-DD)::142",
    "First passage of stool (YYYY-MM-DD)::130",
    "Tolerating solid food (YYYY-MM-DD)::132",
    "Recovered/ready for discharge date (YYYY-MM-DD)::175",
    "Date of discharge (YYYY-MM-DD)::178",
    "Pain control adequate on oral analgesics (YYYY-MM-DD)::155",
    "Termination of urinary drainage (YYYY-MM-DD)::140",
    "Date of follow-up (YYYY-MM-DD)::232",
]

time_cols = [
    "Start of operation time (HH:mm)::71",
    "Stop of operation time (HH:mm)::943",
    "Time of administration of IV antibiotic prophylaxis (HH:mm)::47",
]

anchor_col = "Date of primary operation (YYYY-MM-DD)::53"

ORDINAL_MAPS = {
    "ASA physical status class::78": {
        "ASA 1": 1,
        "ASA 2": 2,
        "ASA 3": 3,
        "ASA 4": 4,
        "Unknown": 5,
        "__MISSING__": 6,
    },
    "Grading of most severe complication::186": {
        "Grade I": 1,
        "Grade II": 2,
        "Grade IIIa": 3,
        "Grade IIIb": 4,
        "Grade IVa": 5,
        "Grade V": 6,
        "Unknown": 7,
        "__MISSING__": 8,
    },
    "Grading of most severe complication::290": {
        "Grade I": 1,
        "Grade II": 2,
        "Grade IIIa": 3,
        "Grade IIIb": 4,
        "Grade V": 5,
        "Unknown": 6,
        "__MISSING__": 7,
    },
    "T - Primary Tumour::223": {
        "Tis": 0,
        "T0": 1,
        "T1": 2,
        "T2": 3,
        "T3": 4,
        "T4": 5,
        "TX": 6,
        "Unknown": 7,
        "__MISSING__": 8,
    },
    "N - Regional Lymph Nodes::224": {
        "N0": 0,
        "N1": 1,
        "N2": 2,
        "NX": 3,
        "Unknown": 4,
        "__MISSING__": 5,
    },
    "M - Distant Metastasis::227": {
        "M0": 0,
        "M1": 1,
        "MX": 2,
        "Unknown": 3,
        "__MISSING__": 4,
    },
}

HIGH_CARDINALITY_NOMINALS = [
    "Main procedure name::56",
    "Additional major procedures::61",
    "Final diagnosis::221",
    "Type of anastomosis::67",
    "Other main postoperative analgesia::146",
]

In [3]:
def _clean_str(s):
    return s.astype("string").str.strip()

def _parse_date(raw):
    s = _clean_str(raw)
    s = s.mask(s.isin(UNKNOWN_TOKENS), pd.NA)
    return pd.to_datetime(s, errors="coerce")

def _parse_time(raw):
    s = _clean_str(raw)
    s = s.mask(s.isin(UNKNOWN_TOKENS), pd.NA)
    return pd.to_datetime(s, format="%H:%M", errors="coerce")

def _make_status(raw):
    s = _clean_str(raw)
    status = pd.Series(0, index=raw.index, dtype="int8")
    status[raw.isna()] = 2
    status[(~raw.isna()) & s.isin(UNKNOWN_TOKENS)] = 1
    return status

def encode_nominal_column(df, col, missing_token=MISSING_TOKEN, min_count_for_other=None):
    s = df[col].astype("string").str.strip().fillna(missing_token)
    if min_count_for_other is not None:
        vc = s.value_counts(dropna=False)
        keep = vc[vc >= min_count_for_other].index
        s = s.where(s.isin(keep), "Other")
    cat = pd.Categorical(s)
    encoded = pd.Series(cat.codes, index=df.index, name=f"{col}__code").astype("int16")
    reverse_map = dict(enumerate(cat.categories))
    return encoded, reverse_map

def encode_ordinal_column(df, col, ordinal_map, missing_token=MISSING_TOKEN):
    s = df[col].astype("string").str.strip().fillna(missing_token)
    unseen = sorted(set(s.unique()) - set(ordinal_map.keys()))
    if unseen:
        raise ValueError(f"{col}: unmapped values found: {unseen}")
    encoded = s.map(ordinal_map).astype("int16")
    reverse_map = {v: k for k, v in ordinal_map.items()}
    return encoded, reverse_map

def find_numeric_like_object_cols(df, threshold=0.9):
    cols = []
    details = []
    obj_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in obj_cols:
        if col in date_cols or col in time_cols:
            continue
        s = df[col].astype("string").str.strip()
        valid = s[~s.isna() & ~s.isin(UNKNOWN_TOKENS)]
        if len(valid) == 0:
            continue
        parsed = pd.to_numeric(valid, errors="coerce")
        ratio = parsed.notna().mean()
        if ratio >= threshold:
            cols.append(col)
            details.append({
                "col": col,
                "valid_non_unknown_count": len(valid),
                "numeric_parse_ratio": round(ratio, 4),
                "unknown_count": int(s.isin(UNKNOWN_TOKENS).sum()),
                "null_count": int(df[col].isna().sum()),
            })
    details_df = pd.DataFrame(details).sort_values(
        ["numeric_parse_ratio", "unknown_count", "null_count"],
        ascending=[False, False, False],
    ) if details else pd.DataFrame(columns=["col", "valid_non_unknown_count", "numeric_parse_ratio", "unknown_count", "null_count"])
    return cols, details_df

def split_numeric_with_status_no_nan(df, col, fill_strategy="median"):
    raw = df[col]
    s = raw.astype("string").str.strip()
    is_missing = raw.isna()
    is_unknown = (~is_missing) & s.isin(UNKNOWN_TOKENS)
    numeric = pd.to_numeric(raw.where(~is_missing & ~is_unknown, np.nan), errors="coerce")
    if fill_strategy == "median":
        fill_value = numeric.median()
    elif fill_strategy == "mean":
        fill_value = numeric.mean()
    elif fill_strategy == "zero":
        fill_value = 0
    else:
        fill_value = fill_strategy
    if pd.isna(fill_value):
        fill_value = 0
    numeric_filled = numeric.fillna(fill_value)
    status = pd.Series(0, index=df.index, dtype="int8")
    status[is_unknown] = 1
    status[is_missing] = 2
    return pd.DataFrame({
        f"{col}__num": numeric_filled,
        f"{col}__status": status,
    }), fill_value

In [4]:
def preprocess_numeric_with_nulls(df_model):
    numeric_missing_meta = {}
    num_cols_with_nulls = [
        c for c in df_model.select_dtypes(include=["number"]).columns
        if df_model[c].isna().any()
    ]
    new_cols = []
    for col in num_cols_with_nulls:
        fill_value = df_model[col].median()
        if pd.isna(fill_value):
            fill_value = 0
        filled = df_model[col].fillna(fill_value)
        missing = df_model[col].isna().astype("int8")
        new_cols.append(filled.rename(col))
        new_cols.append(missing.rename(f"{col}__missing"))
        numeric_missing_meta[col] = {
            "filled_col": col,
            "missing_col": f"{col}__missing",
            "fill_value": float(fill_value),
        }
    if new_cols:
        df_model = df_model.drop(columns=num_cols_with_nulls)
        df_model = pd.concat([df_model, pd.concat(new_cols, axis=1)], axis=1).copy()
    return df_model, numeric_missing_meta

def preprocess_special_numeric_objects(df_model):
    special_numeric_cols, special_numeric_report = find_numeric_like_object_cols(df_model, threshold=0.9)
    special_numeric_meta = {}
    frames = []
    for col in special_numeric_cols:
        new_cols, fill_val = split_numeric_with_status_no_nan(df_model, col, fill_strategy="median")
        decimals = 2
        if any(x in col.lower() for x in ["height", "weight", "temperature", "bmi"]):
            decimals = 2
        elif any(x in col.lower() for x in ["ml", "kcal", "days", "nights", "weeks", "vas"]):
            decimals = 0
        special_numeric_meta[col] = {
            "num_col": f"{col}__num",
            "status_col": f"{col}__status",
            "fill_value": fill_val,
            "decimals": decimals,
        }
        frames.append(new_cols)
    if special_numeric_cols:
        df_model = pd.concat([df_model.drop(columns=special_numeric_cols), pd.concat(frames, axis=1)], axis=1).copy()
    return df_model, special_numeric_meta, special_numeric_cols, special_numeric_report

def preprocess_datetimes(df_model):
    date_time_meta = {"dates": {}, "times": {}, "derived": {}, "anchor": {}}
    additions = []

    anchor_raw = df_model[anchor_col]
    anchor_status = _make_status(anchor_raw)
    anchor_dt = _parse_date(anchor_raw)
    anchor_base = anchor_dt.min()
    anchor_day_index = (anchor_dt - anchor_base).dt.days.astype("float")
    anchor_fill = anchor_day_index.median()
    if pd.isna(anchor_fill):
        anchor_fill = 0.0
    anchor_day_index = anchor_day_index.fillna(anchor_fill)

    additions.append(anchor_day_index.rename(f"{anchor_col}__day_index"))
    additions.append(anchor_status.rename(f"{anchor_col}__status"))

    date_time_meta["anchor"] = {
        "original_col": anchor_col,
        "day_index_col": f"{anchor_col}__day_index",
        "status_col": f"{anchor_col}__status",
        "base_date": str(anchor_base.date()) if pd.notna(anchor_base) else None,
        "fill_value": float(anchor_fill),
    }

    for col in date_cols:
        raw = df_model[col]
        status = _make_status(raw)
        dt = _parse_date(raw)
        if col == anchor_col:
            continue
        offset = (dt - anchor_dt).dt.days.astype("float")
        fill_value = offset.median()
        if pd.isna(fill_value):
            fill_value = 0.0
        offset_filled = offset.fillna(fill_value)
        additions.append(offset_filled.rename(f"{col}__days_from_primary_op"))
        additions.append(status.rename(f"{col}__status"))
        date_time_meta["dates"][col] = {
            "offset_col": f"{col}__days_from_primary_op",
            "status_col": f"{col}__status",
            "fill_value": float(fill_value),
            "anchor_col": anchor_col,
        }

    for col in time_cols:
        raw = df_model[col]
        status = _make_status(raw)
        t = _parse_time(raw)
        minutes = (t.dt.hour * 60 + t.dt.minute).astype("float")
        fill_value = minutes.median()
        if pd.isna(fill_value):
            fill_value = 0.0
        minutes_filled = minutes.fillna(fill_value)
        additions.append(minutes_filled.rename(f"{col}__minutes"))
        additions.append(status.rename(f"{col}__status"))
        date_time_meta["times"][col] = {
            "minutes_col": f"{col}__minutes",
            "status_col": f"{col}__status",
            "fill_value": float(fill_value),
        }

    # derived features
    start_date = _parse_date(df_model["Date of primary operation (YYYY-MM-DD)::53"])
    stop_date = _parse_date(df_model["Stop of operation date (YYYY-MM-DD)::72"])
    start_time = _parse_time(df_model["Start of operation time (HH:mm)::71"])
    stop_time = _parse_time(df_model["Stop of operation time (HH:mm)::943"])
    abx_time = _parse_time(df_model["Time of administration of IV antibiotic prophylaxis (HH:mm)::47"])

    start_dt = start_date + pd.to_timedelta(start_time.dt.hour * 60 + start_time.dt.minute, unit="m")
    stop_dt = stop_date + pd.to_timedelta(stop_time.dt.hour * 60 + stop_time.dt.minute, unit="m")
    cross_midnight = stop_dt < start_dt
    stop_dt.loc[cross_midnight] = stop_dt.loc[cross_midnight] + pd.Timedelta(days=1)

    operation_duration_min = (stop_dt - start_dt).dt.total_seconds() / 60
    op_duration_fill = operation_duration_min.median()
    if pd.isna(op_duration_fill):
        op_duration_fill = 0.0
    additions.append(operation_duration_min.fillna(op_duration_fill).rename("operation_duration_minutes__derived"))

    abx_dt = start_date + pd.to_timedelta(abx_time.dt.hour * 60 + abx_time.dt.minute, unit="m")
    abx_lead_min = (start_dt - abx_dt).dt.total_seconds() / 60
    abx_lead_min.loc[abx_lead_min < 0] = abx_lead_min.loc[abx_lead_min < 0] + 24 * 60
    abx_lead_fill = abx_lead_min.median()
    if pd.isna(abx_lead_fill):
        abx_lead_fill = 0.0
    additions.append(abx_lead_min.fillna(abx_lead_fill).rename("abx_to_op_start_minutes__derived"))

    date_time_meta["derived"] = {
        "operation_duration_minutes__derived": {"fill_value": float(op_duration_fill)},
        "abx_to_op_start_minutes__derived": {"fill_value": float(abx_lead_fill)},
    }

    df_model = pd.concat([df_model.drop(columns=date_cols + time_cols), pd.concat(additions, axis=1)], axis=1).copy()
    return df_model, date_time_meta

def preprocess_categoricals(df_model):
    categorical_meta = {}
    cat_cols = df_model.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    new_encoded_cols = []
    cols_to_drop = []

    for col in cat_cols:
        if col in ORDINAL_MAPS:
            encoded, reverse_map = encode_ordinal_column(df_model, col, ORDINAL_MAPS[col])
        else:
            min_count_for_other = 20 if col in HIGH_CARDINALITY_NOMINALS else None
            encoded, reverse_map = encode_nominal_column(
                df_model, col, missing_token=MISSING_TOKEN, min_count_for_other=min_count_for_other
            )
        encoded_col = f"{col}__code"
        encoded.name = encoded_col
        new_encoded_cols.append(encoded)
        cols_to_drop.append(col)
        categorical_meta[col] = {
            "encoded_col": encoded_col,
            "reverse_map": reverse_map,
            "kind": "ordinal" if col in ORDINAL_MAPS else "nominal",
        }

    if new_encoded_cols:
        df_model = pd.concat([df_model.drop(columns=cols_to_drop), pd.concat(new_encoded_cols, axis=1)], axis=1).copy()
    return df_model, categorical_meta

def preprocess_pipeline(df):
    original_order = df.columns.tolist()
    print(f"original shape: {df.shape}")
    cols_to_drop = list(restore_constants.keys()) + list(restore_copies.keys())
    df_model = df.drop(columns=cols_to_drop).copy()
    print(f"shape after dropping constants and copies: {df_model.shape}")

    df_model, numeric_missing_meta = preprocess_numeric_with_nulls(df_model)
    print(f"shape after numeric with nulls: {df_model.shape}")
    df_model, special_numeric_meta, special_numeric_cols, special_numeric_report = preprocess_special_numeric_objects(df_model)
    print(f"shape after special numeric objects: {df_model.shape}")
    df_model, date_time_meta = preprocess_datetimes(df_model)
    print(f"shape after datetimes: {df_model.shape}")
    df_model, categorical_meta = preprocess_categoricals(df_model)
    print(f"shape after categoricals: {df_model.shape}")

    meta = {
        "original_order": original_order,
        "restore_constants": restore_constants,
        "restore_copies": restore_copies,
        "numeric_missing_meta": numeric_missing_meta,
        "special_numeric_meta": special_numeric_meta,
        "special_numeric_cols": special_numeric_cols,
        "special_numeric_report": special_numeric_report,
        "date_time_meta": date_time_meta,
        "categorical_meta": categorical_meta,
    }
    return df_model, meta

In [5]:
def reverse_categorical_columns(syn_df, categorical_meta, missing_token=MISSING_TOKEN):
    out = syn_df.copy()
    for original_col, meta_entry in categorical_meta.items():
        encoded_col = meta_entry["encoded_col"]
        reverse_map = meta_entry["reverse_map"]

        codes = out[encoded_col].round().astype("Int64")
        min_code = min(reverse_map.keys())
        max_code = max(reverse_map.keys())
        codes = codes.clip(lower=min_code, upper=max_code)

        decoded = codes.map(reverse_map).astype(object)
        decoded[decoded == missing_token] = pd.NA
        out[original_col] = decoded

    helper_cols = [meta["encoded_col"] for meta in categorical_meta.values()]
    out = out.drop(columns=helper_cols, errors="ignore")
    return out


def reverse_date_time_columns(syn_df, date_time_meta):
    out = syn_df.copy()

    anchor_info = date_time_meta["anchor"]
    base_date = pd.to_datetime(anchor_info["base_date"])
    anchor_day_index = out[anchor_info["day_index_col"]].round()
    anchor_status = out[anchor_info["status_col"]].round().clip(0, 2).astype("Int64")

    anchor_rebuilt = base_date + pd.to_timedelta(anchor_day_index, unit="D")
    anchor_rebuilt = anchor_rebuilt.dt.strftime("%Y-%m-%d").astype(object)
    anchor_rebuilt[anchor_status == 1] = "Unknown"
    anchor_rebuilt[anchor_status == 2] = pd.NA

    anchor_col_local = anchor_info["original_col"]
    out[anchor_col_local] = anchor_rebuilt
    anchor_dt = pd.to_datetime(anchor_rebuilt, errors="coerce")

    for original_col, info in date_time_meta["dates"].items():
        offset = out[info["offset_col"]].round()
        status = out[info["status_col"]].round().clip(0, 2).astype("Int64")

        rebuilt = anchor_dt + pd.to_timedelta(offset, unit="D")
        rebuilt = rebuilt.dt.strftime("%Y-%m-%d").astype(object)
        rebuilt[status == 1] = "Unknown"
        rebuilt[status == 2] = pd.NA
        out[original_col] = rebuilt

    for original_col, info in date_time_meta["times"].items():
        mins = out[info["minutes_col"]].round()
        status = out[info["status_col"]].round().clip(0, 2).astype("Int64")

        hh = (mins // 60).astype("Int64")
        mm = (mins % 60).astype("Int64")
        rebuilt = (
            hh.astype("string").str.zfill(2) + ":" +
            mm.astype("string").str.zfill(2)
        ).astype(object)

        rebuilt[status == 1] = "Unknown"
        rebuilt[status == 2] = pd.NA
        out[original_col] = rebuilt

    helper_cols = [
        date_time_meta["anchor"]["day_index_col"],
        date_time_meta["anchor"]["status_col"],
    ]
    for info in date_time_meta["dates"].values():
        helper_cols.extend([info["offset_col"], info["status_col"]])
    for info in date_time_meta["times"].values():
        helper_cols.extend([info["minutes_col"], info["status_col"]])
    helper_cols.extend(list(date_time_meta["derived"].keys()))

    out = out.drop(columns=helper_cols, errors="ignore")
    return out


def rebuild_numeric_with_status(df_syn, meta_entry):
    num_col = meta_entry["num_col"]
    status_col = meta_entry["status_col"]
    decimals = meta_entry.get("decimals", 2)

    num = df_syn[num_col].copy().round(decimals)
    status = df_syn[status_col].copy().round().clip(0, 2).astype("Int64")

    out = num.astype(object)
    out[status == 1] = "Unknown"
    out[status == 2] = pd.NA
    return out


def reverse_special_numeric_columns(syn_df, special_numeric_meta):
    out = syn_df.copy()

    for original_col, meta_entry in special_numeric_meta.items():
        out[original_col] = rebuild_numeric_with_status(out, meta_entry)

    helper_cols = []
    for meta_entry in special_numeric_meta.values():
        helper_cols.extend([meta_entry["num_col"], meta_entry["status_col"]])

    out = out.drop(columns=helper_cols, errors="ignore")
    return out


def reverse_numeric_missing_columns(syn_df, numeric_missing_meta):
    out = syn_df.copy()

    for original_col, meta in numeric_missing_meta.items():
        missing_col = meta["missing_col"]
        miss = out[missing_col].round().clip(0, 1).astype("Int64")
        out.loc[miss == 1, original_col] = pd.NA

    helper_cols = [meta["missing_col"] for meta in numeric_missing_meta.values()]
    out = out.drop(columns=helper_cols, errors="ignore")
    return out


def reverse_pipeline(syn_model, meta, original_df):
    syn_final = syn_model.copy()

    syn_final = reverse_categorical_columns(syn_final, meta["categorical_meta"])
    syn_final = reverse_date_time_columns(syn_final, meta["date_time_meta"])
    syn_final = reverse_special_numeric_columns(syn_final, meta["special_numeric_meta"])
    syn_final = reverse_numeric_missing_columns(syn_final, meta["numeric_missing_meta"])

    # Restore dropped all-null columns with original dtype
    for col in meta["restore_constants"]:
        syn_final[col] = np.nan
        syn_final[col] = syn_final[col].astype(original_df[col].dtype)

    # Restore exact duplicate columns
    for col, source_col in meta["restore_copies"].items():
        syn_final[col] = syn_final[source_col]

    # Restore exact original order
    syn_final = syn_final.reindex(columns=meta["original_order"])

    return syn_final

In [6]:
df = pd.read_csv("data.csv")
df_model, meta = preprocess_pipeline(df)
print(df_model.shape)


C:\Users\user\AppData\Local\Temp\ipykernel_5128\1779834555.py:1: DtypeWarning: Columns (79,154,155,212,228,233,248,249,250,252,253,254,255,256) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data.csv")


original shape: (8000, 286)
shape after dropping constants and copies: (8000, 265)
shape after numeric with nulls: (8000, 288)
shape after special numeric objects: (8000, 325)
shape after datetimes: (8000, 346)
shape after categoricals: (8000, 346)
(8000, 346)


In [7]:
df_model.to_csv("preprocessed.csv", index=False)

In [8]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from numpy.linalg import LinAlgError


def generate_gaussian_copula(df_model, n_samples=None, random_state=42, jitter=1e-6):
    """Gaussian copula generator for all-numeric data."""
    rng = np.random.default_rng(random_state)
    if n_samples is None:
        n_samples = len(df_model)
    X = df_model.copy()
    if not all(np.issubdtype(dt, np.number) for dt in X.dtypes):
        raise ValueError("df_model must contain only numeric columns")
    X_jittered = X.astype(float).copy()
    for col in X_jittered.columns:
        if X_jittered[col].nunique(dropna=False) > 1:
            X_jittered[col] = X_jittered[col] + rng.normal(0, jitter, size=len(X_jittered))
    U = pd.DataFrame(index=X.index)
    for col in X_jittered.columns:
        ranks = X_jittered[col].rank(method="average")
        U[col] = (ranks - 0.5) / len(X_jittered)
    Z = pd.DataFrame(norm.ppf(U.clip(1e-6, 1 - 1e-6)), columns=X.columns)
    corr = Z.corr().to_numpy()
    if np.any(np.isnan(corr)) or np.any(np.isinf(corr)):
        corr = np.eye(corr.shape[0])
    corr = np.clip(corr, -1, 1)
    eps = 1e-3
    corr = corr + np.eye(corr.shape[0]) * eps
    eigenvals, eigenvecs = np.linalg.eigh(corr)
    eigenvals[eigenvals < 0] = 0
    corr = eigenvecs @ np.diag(eigenvals) @ eigenvecs.T
    corr = (corr + corr.T) / 2
    try:
        z_syn = rng.multivariate_normal(mean=np.zeros(len(X.columns)), cov=corr, size=n_samples)
    except Exception as e:
        print(f"Error in multivariate normal: {e}, using independent normal sampling")
        z_syn = rng.normal(size=(n_samples, len(X.columns)))
    z_syn = pd.DataFrame(z_syn, columns=X.columns)
    u_syn = pd.DataFrame(norm.cdf(z_syn), columns=X.columns)
    syn = pd.DataFrame(index=range(n_samples))
    for col in X.columns:
        sorted_vals = np.sort(X[col].to_numpy())
        q = u_syn[col].to_numpy()
        idx = np.floor(q * (len(sorted_vals) - 1)).astype(int)
        idx = np.clip(idx, 0, len(sorted_vals) - 1)
        syn[col] = sorted_vals[idx]
    return syn


In [9]:
syn_model = generate_gaussian_copula(df_model, n_samples=len(df_model), random_state=42)
print(syn_model.shape)
syn_model.head()

C:\Users\user\AppData\Local\Temp\ipykernel_5128\2201424905.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  U[col] = (ranks - 0.5) / len(X_jittered)
C:\Users\user\AppData\Local\Temp\ipykernel_5128\2201424905.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  U[col] = (ranks - 0.5) / len(X_jittered)
C:\Users\user\AppData\Local\Temp\ipykernel_5128\2201424905.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joinin

(8000, 346)


,Year of Birth::18,Year of Birth::18__missing,Age::40,Age::40__missing,BMI::24,BMI::24__missing,Last HbA1c value ((mmol/mol))::28,Last HbA1c value ((mmol/mol))::28__missing,Days between admission and the last chemotherapy::30,Days between admission and the last chemotherapy::30__missing,Total IV volume of fluids intra-operatively (ml)::101,Total IV volume of fluids intra-operatively (ml)::101__missing,Total IV volume of fluids day zero (ml)::107,Total IV volume of fluids day zero (ml)::107__missing,Weight change day 1 (kg)::112,Weight change day 1 (kg)::112__missing,Weight change day 2 (kg)::116,Weight change day 2 (kg)::116__missing,Weight change day 3 (kg)::117,Weight change day 3 (kg)::117__missing,Time to passage of flatus (nights)::129,Time to passage of flatus (nights)::129__missing,Time to passage of stool (nights)::131,Time to passage of stool (nights)::131__missing,Time to tolerating solid food (nights)::133,Time to tolerating solid food (nights)::133__missing,Time to termination of urinary drainage (nights)::141,Time to termination of urinary drainage (nights)::141__missing,Time to recovery of ADL ability (nights)::143,Time to recovery of ADL ability (nights)::143__missing,Time to termination of epidural analgesia (nights)::149,Time to termination of epidural analgesia (nights)::149__missing,Time to pain control with oral analgesics (nights)::156,Time to pain control with oral analgesics (nights)::156__missing,Length of stay (nights in hospital after primary operation)::179,Length of stay (nights in hospital after primary operation)::179__missing,Number of nights receiving intensive care::184,Number of nights receiving intensive care::184__missing,Time between operation and follow-up (nights)::235,Time between operation and follow-up (nights)::235__missing,Number of nights receiving intensive care::284,Number of nights receiving intensive care::284__missing,Length of stay for readmissions::354,Length of stay for readmissions::354__missing,Total length of stay (nights)::353,Total length of stay (nights)::353__missing,Weight 6 months prior to admission (kg)::21__num,Weight 6 months prior to admission (kg)::21__status,Preoperative body weight (kg)::20__num,Preoperative body weight (kg)::20__status,Height (cm)::23__num,Height (cm)::23__status,Termination of smoking (no. of weeks before surgery)::25__num,Termination of smoking (no. of weeks before surgery)::25__status,Standard units per week::41__num,Standard units per week::41__status,Termination of alcohol (no of weeks before surgery)::26__num,Termination of alcohol (no of weeks before surgery)::26__status,Distance from anal verge::1840__num,Distance from anal verge::1840__status,Length of incision (cm)::64__num,Length of incision (cm)::64__status,Intraoperative blood loss (ml)::69__num,Intraoperative blood loss (ml)::69__status,Core body temperature at end of operation (°C)::95__num,Core body temperature at end of operation (°C)::95__status,IV volume of crystalloids intraoperatively (ml)::97__num,IV volume of crystalloids intraoperatively (ml)::97__status,IV volume of colloids intraoperatively (ml)::99__num,IV volume of colloids intraoperatively (ml)::99__status,IV volume of blood products intra-operatively (ml)::100__num,IV volume of blood products intra-operatively (ml)::100__status,"Intravenous fluids, volume infused - On day of surgery, postoperatively (ml)::106__num","Intravenous fluids, volume infused - On day of surgery, postoperatively (ml)::106__status",Duration of IV fluid infusion (nights)::109__num,Duration of IV fluid infusion (nights)::109__status,Morning weight - On postoperative day 1 (kg)::111__num,Morning weight - On postoperative day 1 (kg)::111__status,Morning weight - On postoperative day 2 (kg)::113__num,Morning weight - On postoperative day 2 (kg)::113__status,Morning weight - On postoperative day 3 (kg)::114__num,Morning weight - On postoperative day 3 (kg)::114__status,"Oral fluids, total volume taken - On day of surgery, postoperatively (ml)::

In [10]:
#
# # Train / sample your generator on df_model here
# # syn_model = generator.sample(len(df_model))
#

In [11]:

# Reverse back to original schema
syn_final = reverse_pipeline(syn_model, meta, df)
print(syn_final.shape)

C:\Users\user\AppData\Local\Temp\ipykernel_5128\179346338.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[original_col] = decoded
C:\Users\user\AppData\Local\Temp\ipykernel_5128\179346338.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[original_col] = decoded
C:\Users\user\AppData\Local\Temp\ipykernel_5128\179346338.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using p

(8000, 286)


In [12]:
syn_final.to_csv('output3.csv', index=False)

In [13]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
from scipy.stats import ks_2samp, chisquare
from sklearn.metrics import pairwise_distances

def evaluate_synthetic_data(df, syn_final, df_model, syn_model, random_state=42):
    results = {}

    # -------------------------------------------------
    # 1) DATA STRUCTURE
    # -------------------------------------------------
    structure = {
        "same_shape_final": df.shape == syn_final.shape,
        "same_columns_final": list(df.columns) == list(syn_final.columns),
        "dtype_match_rate_final": float((df.dtypes.astype(str) == syn_final.dtypes.astype(str)).mean()),
        "original_shape": df.shape,
        "synthetic_final_shape": syn_final.shape,
    }
    results["data_structure"] = structure


    # PRIVACY
    X_real = df_model.copy()
    X_syn = syn_model.copy()

    common_cols = [c for c in X_real.columns if c in X_syn.columns]
    X_real = X_real[common_cols]
    X_syn = X_syn[common_cols]

    X_real = X_real.fillna(X_real.median(numeric_only=True))
    X_syn = X_syn.fillna(X_real.median(numeric_only=True))

    # compute pairwise distances directly
    dists = pairwise_distances(X_syn, X_real, metric="euclidean")
    min_dists = dists.min(axis=1)

    privacy = {
        "mean_nn_distance_syn_to_real": float(np.mean(min_dists)),
        "min_nn_distance_syn_to_real": float(np.min(min_dists)),
        "median_nn_distance_syn_to_real": float(np.median(min_dists)),
    }
    results["privacy"] = privacy

    # -------------------------------------------------
    # 3) DISCRIMINATION
    # Can a classifier distinguish real from synthetic?
    # -------------------------------------------------
    X_combined = pd.concat([X_real, X_syn], axis=0).reset_index(drop=True)
    y_combined = np.array([0] * len(X_real) + [1] * len(X_syn))

    clf = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=random_state,
            n_jobs=-1
        ))
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X_combined, y_combined, test_size=0.3, random_state=random_state, stratify=y_combined
    )

    clf.fit(X_train, y_train)
    proba = clf.predict_proba(X_test)[:, 1]
    pred = clf.predict(X_test)

    discrimination = {
        "auc_real_vs_synthetic": float(roc_auc_score(y_test, proba)),
        "accuracy_real_vs_synthetic": float(accuracy_score(y_test, pred)),
    }
    results["discrimination"] = discrimination

    # -------------------------------------------------
    # 4) DISTRIBUTION
    # Numeric: KS test
    # Categorical: frequency difference on final data
    # -------------------------------------------------
    dist_summary = {}

    # numeric distribution in processed space
    numeric_cols = X_real.select_dtypes(include=["number"]).columns
    ks_rows = []
    for col in numeric_cols:
        a = X_real[col].dropna()
        b = X_syn[col].dropna()
        if len(a) > 0 and len(b) > 0:
            ks_stat, ks_p = ks_2samp(a, b)
            ks_rows.append((col, ks_stat, ks_p))
    ks_df = pd.DataFrame(ks_rows, columns=["column", "ks_stat", "ks_pvalue"]).sort_values("ks_stat", ascending=False)

    dist_summary["numeric_ks_mean"] = float(ks_df["ks_stat"].mean()) if not ks_df.empty else np.nan
    dist_summary["numeric_ks_top10"] = ks_df.head(10)

    # categorical distribution on reconstructed final data
    cat_cols = df.select_dtypes(include=["object", "string", "category"]).columns
    cat_rows = []
    for col in cat_cols:
        real_freq = df[col].astype("string").fillna("__MISSING__").value_counts(normalize=True)
        syn_freq = syn_final[col].astype("string").fillna("__MISSING__").value_counts(normalize=True)

        all_levels = sorted(set(real_freq.index).union(set(syn_freq.index)))
        real_vec = np.array([real_freq.get(x, 0.0) for x in all_levels])
        syn_vec = np.array([syn_freq.get(x, 0.0) for x in all_levels])

        l1_diff = np.abs(real_vec - syn_vec).sum()
        cat_rows.append((col, l1_diff))

    cat_df = pd.DataFrame(cat_rows, columns=["column", "l1_freq_diff"]).sort_values("l1_freq_diff", ascending=False)

    dist_summary["categorical_l1_mean"] = float(cat_df["l1_freq_diff"].mean()) if not cat_df.empty else np.nan
    dist_summary["categorical_top10"] = cat_df.head(10)

    results["distribution"] = dist_summary

    return results

In [14]:
results = evaluate_synthetic_data(df, syn_final, df_model, syn_model)

print("DATA STRUCTURE")
print(results["data_structure"])

print("\nPRIVACY")
print(results["privacy"])

print("\nDISCRIMINATION")
print(results["discrimination"])

print("\nDISTRIBUTION")
print("Mean numeric KS:", results["distribution"]["numeric_ks_mean"])
print(results["distribution"]["numeric_ks_top10"])

print("\nMean categorical L1 diff:", results["distribution"]["categorical_l1_mean"])
print(results["distribution"]["categorical_top10"])

C:\Users\user\AppData\Local\Temp\ipykernel_5128\2068879433.py:92: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  ks_stat, ks_p = ks_2samp(a, b)


DATA STRUCTURE
{'same_shape_final': True, 'same_columns_final': True, 'dtype_match_rate_final': 1.0, 'original_shape': (8000, 286), 'synthetic_final_shape': (8000, 286)}

PRIVACY
{'mean_nn_distance_syn_to_real': 1158.9011205182733, 'min_nn_distance_syn_to_real': 448.87498796997, 'median_nn_distance_syn_to_real': 1127.5389630080153}

DISCRIMINATION
{'auc_real_vs_synthetic': 0.999996875, 'accuracy_real_vs_synthetic': 0.9985416666666667}

DISTRIBUTION
Mean numeric KS: 0.003595736994219654
                                                column   ks_stat  ks_pvalue
62             Intraoperative blood loss (ml)::69__num  0.019375   0.099251
158                operation_duration_minutes__derived  0.016375   0.233741
302                N - Regional Lymph Nodes::224__code  0.015125   0.319479
220                    Artificial nutrition::134__code  0.014000   0.413169
226      Other main postoperative analgesia::146__code  0.013625   0.447713
218    Intravenous fluid infusion restarted::110__cod